This sample shows how to run inference on multiple video sources. \
All decoded frames will be sent to YOLO 11s for object detection.

In [ ]:
import python_vali as vali

import torch
from ultralytics import YOLO

import threading
import nvtx
import queue
import time
from PIL import Image
from IPython.display import display, clear_output
import cv2

In [ ]:
class StopExecution(Exception):
    def _render_traceback_(self):
        return []

There are 2 modes of operation:
  - demo mode: input will be limited by 1 source, detection results will be plotted and shown
  - perf mode: all given input sources will be processed simultaneously, inference results will not be processed

In [ ]:
# In demo mode, only one stream will be processed, and inderence resuls will be
# visualised.
# Otherwise, batch inference will be done, results won't undergo any
# postprocessing for the sake of performance.
DEMO_MODE = True

# Input video URLs.
urls = [
    "../tests/data/test.mp4",
]

# NN model
model = YOLO("yolo11s.pt")
model.eval()
model.to("cuda")

try:
    # Video decoders pool
    pyDecs = []
    for url in urls:
        # Use big packet queue size because in demo mode postprocessing may be
        # quite slow. We don't want reader thread to fail to push the packet
        # because main thread is busy drawing bboxes.
        pyDecs.append(vali.PyDecoder(url, {}, gpu_id=0, pkt_queue_size=75))
        if DEMO_MODE:
            break

    # Combined conversion + rescale
    pyUD = vali.PySurfaceUD(gpu_id=0, stream=pyDecs[0].Stream)

except Exception as e:
    print(repr(e))
    raise StopExecution()

# Video frame width and height as model likes it.
# Keep original aspect ratio, make height multiple of 32.
target_w = 640
target_h = int(pyDecs[0].Height / pyDecs[0].Width * target_w) // 32 * 32


# Model batch size
batch_size = 4

# Video frames and events queue for sync between decoding and inference streams
inf_queue = queue.Queue(maxsize=batch_size)

`ReaderContext` class is used to read compressed video packets in separate thread and put them into decoder internal queue. \
When working with network inputs, it's better to dedicate separate thread to this task because otherwise a network timeout will affect other readers.

In [ ]:
class ReaderContext:
    """
    This class is used to read compressed packets from video source.
    """

    def __init__(self, source_id: int):
        """
        Constructor.

        Args:
            source_id (int): video source id
        """
        self.source_id = source_id
        self.pyDec = pyDecs[self.source_id]
        self.pkt_id = 0
        self.worker = threading.Thread(
            target=self.read_packet, name=f"reader_thread_{source_id}")

    def start(self):
        """
        Start reader thread.
        """
        self.worker.start()

    def join(self):
        """
        Join reader thread.
        """
        self.worker.join()

    def read_packet(self) -> None:
        """
        Reads packet into decoder internal queue.
        """

        @nvtx.annotate()
        def _read_pkt(pyDec: vali.PyDecoder) -> vali.DecodeStatus:
            return pyDec.ReadPacket()

        while True:
            status = _read_pkt(self.pyDec)
            if status in [vali.DecodeStatus.OVER, vali.DecodeStatus.ERROR]:
                return
            self.pkt_id += 1

            if DEMO_MODE:
                time.sleep(1.0 / self.pyDec.Framerate)

`PreprocessingContext` class runs actual decoding and color conversion. Result is pushed to queue. \
Nvdec is a separate hw unit and decoding process is async by it's nature, one thread is generally sufficient to saturate single nvdec.

In [ ]:
class PreprocessingContext:
    """
    This class is used to decode and preprocess video frames for futher inference.
    """

    def __init__(self, inf_queue: queue.Queue, source_id: int):
        """
        Constructor

        Args:
            inf_queue (queue.Queue): preprocessed frames queue
            source_id (int): video source id
        """
        self.source_id = source_id
        self.frame_id = 0
        self.active = True
        self.inf_queue = inf_queue
        self.pyDec = pyDecs[self.source_id]
        self.event = vali.CudaStreamEvent(self.pyDec.Stream, gpu_id=0)
        self.start_time = time.time()

        self.surf_dec = vali.Surface.Make(
            format=self.pyDec.Format,
            width=self.pyDec.Width,
            height=self.pyDec.Height,
            gpu_id=0)

    @nvtx.annotate()
    def decode_to_tensor(self, surf_inf: vali.Surface) -> vali.DecodeStatus:
        """
        Decode and preprocess video frame in-place.
        Outputs performance stats every 3 seconds.

        Args:
            surf_inf (vali.Surface): output surface

        Returns:
            vali.DecodeStatus: opeation status
        """
        status = self.pyDec.DecodePacketToSurfaceAsync(self.surf_dec)
        if status != vali.DecodeStatus.SUCCESS:
            return status

        success, details = pyUD.RunAsync(self.surf_dec, surf_inf)
        if not success:
            print(details)
            return vali.DecodeStatus.ERROR

        # It's safe to reuse CUDA events
        self.event.Record()

        # We have enough Surfaces allocated to support continuous operations.
        # Hence there's no nedd to copy memory between Surface and tensor.
        # This tensor will serve as a wrapper object.
        img_tensor = torch.from_dlpack(surf_inf).clamp(0.0, 1.0)
        img_tensor = torch.reshape(img_tensor, [3, target_h, target_w])

        # Queue has fixed size, so if this NVTX marker is growing larger it
        # only means that preprocessing thread is waiting for inference thread
        # to take frames from queue.
        with nvtx.annotate("PutIntoQueue"):
            self.inf_queue.put((img_tensor, self.event))

        self.frame_id += 1

        # Output perf stats
        if not DEMO_MODE and self.frame_id % (3 * int(self.pyDec.Framerate)) == 0:
            clear_output(wait=True)
            fps = int(self.frame_id / (time.time() - self.start_time))
            print(f"source {self.source_id}: {fps} fps")

        return vali.DecodeStatus.SUCCESS

    @staticmethod
    def process_sources(contexts: list):
        """
        Static method which checks decoders in round-robin fashon. If there's a
        decoded frame, it's preprocessed and put into queue.

        It also allocates Surfaces for inference.

        Args:
            contexts (list): list of preprocessing contexts.
        """
        # Max memory consumption is `batch_size` tensors processed by model + 
        # `inf_queue.maxsize` tensors in the queue.
        surf_inf = [vali.Surface.Make(
            format=vali.PixelFormat.RGB_32F_PLANAR,
            width=target_w,
            height=target_h,
            gpu_id=0) for _ in range(0, batch_size + inf_queue.maxsize)]

        while True:
            num_active_ctx = len(contexts)
            idx = 0

            for ctx in contexts:
                if not ctx.active:
                    num_active_ctx -= 1
                    continue

                status = ctx.decode_to_tensor(surf_inf[idx])
                if status == vali.DecodeStatus.DONE or status == vali.DecodeStatus.ERROR:
                    ctx.active = False
                    print(f"{ctx.frame_id} frames processed\n")

                idx = ctx.frame_id % len(surf_inf)

            if num_active_ctx == 0:
                return

`InferenceContext` class is used to take preprocessed frames from queue and run inference on them. \
It's done by separate thread, because otherwise decoding and inference will be done in serial fashion whin will limit hw utilization.

In [ ]:
class InferenceContext:
    """
    This class is used to run inference on video frames.
    """

    def __init__(self):
        self.stream = torch.cuda.Stream()

    @nvtx.annotate()
    def _output(self, results) -> None:
        """
        Post process inference results
        """
        for result in results:
            if result.boxes:
                annotated_img = result.plot()
                rgb_img = cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB)
                # Show in the same cell
                clear_output(wait=True)
                display(Image.fromarray(rgb_img), display_id="frame")

    @nvtx.annotate()
    def run_inference(self, frames: list[torch.tensor]) -> None:
        """
        Run inference.

        Args:
            frames (list[torch.tensor]): list of video frames.
        """
        if DEMO_MODE:
            # 'public' model API doesn't support tensor batch as input.
            # It causes suboptimal CUDA SM utilization.
            for frame in frames:
                frame = torch.unsqueeze(frame, dim=0)
                results = model(frame, verbose=False, conf=0.75)
                self._output(results)
        else:
            # Please note that model can accept tensor batch only via it's
            # 'private' API which doesn't do any postptocessing and outputs
            # raw tesor with inference results.
            batch = torch.stack(frames, dim=0)
            with torch.cuda.stream(self.stream):
                results = model.model(batch)

Main loop. \
Init readers, start them. Then start decoder and inference thread, then join threads when done.

In [ ]:
# Init readers first because it may take some time in case of network input.
readers = []
for source_id in range(0, len(pyDecs)):
    readers.append(ReaderContext(source_id))

# Start reader threads.
for reader in readers:
    reader.start()

# Init preprocessing.
preproc_ctx = []
for source_id in range(0, len(readers)):
    preproc_ctx.append(PreprocessingContext(inf_queue, source_id))

# Start preprocessing thread
preproc_thread = threading.Thread(
    target=PreprocessingContext.process_sources, name=f"preproc_thread", kwargs={'contexts': preproc_ctx})
preproc_thread.start()

# Run inference in main thread
inf_ctx = InferenceContext()
while True:
    try:
        # If no frames arrive within timeout, consider job done.
        timeout_s = 3.0
        frames = []
        events = []
        while len(frames) < batch_size:
            (frame, event) = inf_queue.get(timeout=timeout_s)
            frames.append(frame)
            events.append(event)

        # Postpone the events wait as long as possible in hope majority of them
        # will be done when it's time to run inference.
        for event in events:
            event.Wait()

        inf_ctx.run_inference(frames)
    except queue.Empty:
        break

# Join threads
for reader in readers:
    reader.join()

preproc_thread.join()